# Machine Learning - Practical 10: End-to-End ML Case Study / Project Clinic

**Course:** Machine Learning - National University of Kyiv-Mohyla Academy (NaUKMA)

**Instructor:** Dmytro Kuzmenko - kuzmenko@ukma.edu.ua

| | |
|---|---|
| Week | 10 |
| Module | 10. Machine Learning in Practice |
| Format | Practical session (not graded - exam preparation) |
| Estimated time | 1.5-2 h of active work including discussion |
| Prerequisites | P01-P09; mini-project specification (materials/project/specification.md) |


## AI Use Disclosure

Fill this in before submitting (see course policy).

| Field | Your entry |
|---|---|
| AI tools used | |
| Nature of assistance | |
| Representative prompts or relevant interaction | |
| What I independently verified or changed | |

## Learning Objectives

- Run a complete, compact ML workflow end-to-end on a real small dataset: problem, baseline, preprocessing pipeline, model comparison, CV, metrics, error analysis, conclusions.
- Map every workflow step to the 11-step mini-project methodology checklist.
- Design a valid validation strategy for a grouped dataset (two scenarios) and justify it in words and code.

## Warm-up (10 min)

### Question 1 (multiple choice)

You receive a new tabular dataset and a business question. What is the first methodological step?

- A. Train the best available model as fast as possible to see the maximum accuracy.
- B. Formulate the prediction problem precisely (target, unit of prediction, evaluation cost, deployment context) and inspect the data.
- C. Download more data from the internet to be safe.
- D. Split the data into train and test and tune a neural network on the test set.


**Your answer:**

### Question 2 (quick reasoning)

Why do we build a baseline (majority class, mean prediction, or a simple linear model) before comparing sophisticated models?


**Your answer:**

### Question 3 (mini-interpretation)

A model reaches accuracy 0.97; the majority-class baseline reaches 0.63. What does this comparison tell you, and which metric would you additionally want for this binary problem?


**Your answer:**

## Guided Exercise (70 min)

### Task 1: Complete end-to-end workflow

One compact end-to-end workflow on the breast_cancer dataset (30 features, 569 samples, binary target: malignant/benign). Each step is labeled; the numbers you produce are the evidence. All experiments are deterministic.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, f1_score,
                             roc_auc_score, roc_curve)
%matplotlib inline
np.random.seed(42)

### Step 1: Problem formulation

Predict whether a tumor is malignant or benign from cell-nucleus measurements. The decision is a screening aid: the cost of missing a malignant case is high, so recall for the malignant class matters, not only accuracy.

**Task.** Write the problem in one sentence in your own words, and state which error is more expensive (false negative vs false positive) for this use case.


**Your answer:**

### Step 2: Data inspection

**Experiment.** Load the data and inspect shape, feature names, target distribution, missing values, and the scale of a few features.

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target
print("shape:", X.shape)
print("classes (0=malignant, 1=benign):", pd.Series(y).value_counts().to_dict())
print("missing values:", int(np.isnan(X).sum()))
print("feature names:", list(data.feature_names[:5]), "...")
scale = pd.DataFrame(X, columns=data.feature_names).std().sort_values()
print("feature std, min:", scale.iloc[0], " max:", scale.iloc[-1])

shape: (569, 30)
classes (0=malignant, 1=benign): {1: 357, 0: 212}
missing values: 0
feature names: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness'] ...
feature std, min: 0.002646070967089194  max: 569.3569926699492


**Interpretation.** What do the class proportions and the spread of feature scales imply for (a) the choice of metric and (b) the preprocessing steps?


**Your answer:**

### Step 3: Baseline

**Experiment.** A majority-class dummy classifier evaluated with the same cross-validation protocol as the real models.

In [3]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
baseline = Pipeline([("sc", StandardScaler()),
                     ("m", DummyClassifier(strategy="most_frequent"))])
res_base = cross_validate(baseline, X, y, cv=cv, scoring=["accuracy", "f1", "roc_auc"])
print("baseline accuracy: {:.4f}   f1: {:.4f}   roc_auc: {:.4f}".format(
    res_base["test_accuracy"].mean(), res_base["test_f1"].mean(), res_base["test_roc_auc"].mean()))

baseline accuracy: 0.6274   f1: 0.7711   roc_auc: 0.5000


**Interpretation.** Why is the baseline F1 high (about 0.77) despite predicting only one class? Which metric is immune to this illusion?


**Your answer:**

### Step 4-6: Preprocessing pipeline, model candidates, validation scheme

**Decision.** All models are wrapped in a Pipeline (StandardScaler + model) and evaluated with StratifiedKFold(5), so no preprocessing sees the validation folds. Candidate models: logistic regression, RBF SVM, random forest.

In [4]:
models = {
    "LogisticRegression": Pipeline([("sc", StandardScaler()),
                                      ("m", LogisticRegression(max_iter=3000, C=1.0))]),
    "SVC (RBF)": Pipeline([("sc", StandardScaler()),
                            ("m", SVC(C=1.0, gamma="scale", probability=True))]),
    "RandomForest": Pipeline([("sc", StandardScaler()),
                               ("m", RandomForestClassifier(n_estimators=200, random_state=42))]),
}
rows = []
fitted = {}
for name, pipe in models.items():
    res = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1", "roc_auc"])
    rows.append({"model": name,
                 "CV accuracy": round(res["test_accuracy"].mean(), 4),
                 "CV F1": round(res["test_f1"].mean(), 4),
                 "CV ROC-AUC": round(res["test_roc_auc"].mean(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

             model  CV accuracy  CV F1  CV ROC-AUC
LogisticRegression       0.9737 0.9794      0.9953
         SVC (RBF)       0.9772 0.9820      0.9945
      RandomForest       0.9543 0.9637      0.9896


**Interpretation.** Which model wins on which metric? Are the differences large enough to matter, given the fold-to-fold variability you saw in P06?


**Your answer:**

### Step 7: Metrics on a held-out test set

**Decision.** The three models are nearly tied; we choose LogisticRegression because it has the best CV ROC-AUC and produces well-calibrated probabilities, which suit the screening use case. **Experiment.** Split the data once (75/25, stratified), retrain this model on the training part only, and report test accuracy, F1, ROC-AUC, the confusion matrix, and the ROC curve.

In [5]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
best = models["LogisticRegression"]
best.fit(Xtr, ytr)
pred = best.predict(Xte)
prob = best.predict_proba(Xte)[:, 1]
print("test accuracy : {:.4f}".format((pred == yte).mean()))
print("test F1       : {:.4f}".format(f1_score(yte, pred)))
print("test ROC-AUC  : {:.4f}".format(roc_auc_score(yte, prob)))

test accuracy : 0.9860
test F1       : 0.9889
test ROC-AUC  : 0.9977


In [6]:
disp = ConfusionMatrixDisplay(confusion_matrix(yte, pred),
                              display_labels=["malignant", "benign"])
disp.plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix on the held-out test set")
plt.tight_layout()

In [7]:
fpr, tpr, _ = roc_curve(yte, prob)
plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, lw=2, label="LogisticRegression (AUC = {:.3f})".format(roc_auc_score(yte, prob)))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="random")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
plt.tight_layout()

**Interpretation.** Which metric would you report to the stakeholder for this problem, and how do the confusion-matrix numbers map to the false-negative cost you identified in Step 1?


**Your answer:**

### Step 8: Error analysis

**Experiment.** The best model makes very few errors on the test set. Inspect the misclassified samples and, because there are so few, also the most uncertain predictions (probability closest to 0.5): errors concentrate near the decision boundary.

In [8]:
mis = np.where(pred != yte)[0]
print("misclassified:", len(mis))
for i in mis:
    print("  sample {}: true={} pred={} P(benign)={:.3f}".format(
        i, data.target_names[yte[i]], data.target_names[pred[i]], prob[i]))

misclassified: 2
  sample 111: true=benign pred=malignant P(benign)=0.383
  sample 135: true=malignant pred=benign P(benign)=0.893


In [9]:
uncertain = np.argsort(np.abs(prob - 0.5))[:6]
print("most uncertain predictions:")
for i in uncertain:
    ok = "correct" if pred[i] == yte[i] else "WRONG"
    print("  sample {}: true={} pred={} P(benign)={:.3f}  {}".format(
        i, data.target_names[yte[i]], data.target_names[pred[i]], prob[i], ok))

most uncertain predictions:
  sample 92: true=benign pred=benign P(benign)=0.555  correct
  sample 2: true=benign pred=benign P(benign)=0.559  correct
  sample 138: true=benign pred=benign P(benign)=0.592  correct
  sample 116: true=benign pred=benign P(benign)=0.599  correct
  sample 111: true=benign pred=malignant P(benign)=0.383  WRONG
  sample 113: true=malignant pred=malignant P(benign)=0.363  correct


**Interpretation.** Where do the errors live (probability values), and what does that imply about collecting more features or a different decision threshold for this screening task?


**Your answer:**

### Step 9: Conclusions and limitations

**Task.** Write 2-3 sentences: what the experiment establishes, what it does not establish (e.g., deployment on a different hospital, prevalence changes, patient groups), and what you would do next.


**Your answer:**

## Discussion: Mini-project clinic (20 min)

The mini-project is graded on methodology, not on beating a benchmark. Walk through the 11-step checklist from the specification and mark which steps you already completed today:

1. Formulate a problem/question.
2. Inspect the data (size, types, missing values, target distribution).
3. Build a baseline.
4. Choose preprocessing with no leakage (inside a pipeline/CV).
5. Choose 1-3 models with justification.
6. Choose the validation scheme valid for your data (KFold, stratified, grouped, time-aware).
7. Choose metrics appropriate to the task and class distribution.
8. Compare experiments with tables, curves, statistics - not a single number.
9. Perform error analysis (where the model fails and why).
10. State limitations and conclusions.
11. Make the work reproducible (seeds, top-to-bottom notebook, documented dependencies).

### Discussion questions

1. Which of the 11 steps do students most often skip, and what goes wrong later because of that?
2. Your project has a grouped structure (several records per person). Which checklist items change compared with today's breast_cancer workflow, and which stay the same?
3. A classmate reports "my model got 0.97, better than the paper's 0.95". What three questions do you ask before believing the comparison?
4. The project defense lasts 5-7 minutes and every author must answer individually. What is the fastest way to fail the defense despite a good notebook?
5. What must the AI-use disclosure contain for the mini-project, and why is "I used Copilot for the code" insufficient?

## Challenge (30 min)

### Task 2: Design the validation strategy for a grouped dataset

Choose **one** of the two scenarios, describe the validation strategy in words, and write a small code sketch that implements it. Do not train any model - the deliverable is the strategy.

**Scenario A - patient sensor data.** 50 patients, 10 recordings each (500 rows). Goal: predict whether a patient has a condition. Recordings from the same patient are highly correlated.

**Scenario B - retail sales.** Daily sales for 3 years for 20 stores. Goal: predict next-week sales per store. Sales are seasonal and trend over time; stores differ in scale.

**Your strategy (words).** For your chosen scenario: what is the unit of independence, which splitter matches it, how many folds, and what would you report (per-fold scores, aggregated how)?


**Your answer:**

**Your code sketch.** The cell below generates a miniature version of the scenario data. Write the splitter call (and, for Scenario B, the ordering of splits) that implements your strategy.

In [10]:
# --- miniature data for the scenario you chose ---
rng = np.random.RandomState(42)
if False:  # Scenario A: 50 patients x 10 recordings
    n_patients, n_rec = 50, 10
    patient_id = np.repeat(np.arange(n_patients), n_rec)
    X_a = rng.randn(n_patients * n_rec, 5) + patient_id[:, None] * 0.1
    y_a = (rng.rand(n_patients) > 0.5).repeat(n_rec)  # one label per patient
    # from sklearn.model_selection import GroupKFold
    # splitter = GroupKFold(n_splits=5)
    # for tr, te in splitter.split(X_a, y_a, groups=patient_id): ...
else:  # Scenario B: 20 stores x 1096 days
    n_stores, n_days = 20, 1096
    store_id = np.repeat(np.arange(n_stores), n_days)
    day = np.tile(np.arange(n_days), n_stores)
    X_b = rng.randn(n_stores * n_days, 3)
    y_b = (day / 7) % 52 + rng.randn(n_stores * n_days)  # weekly seasonality
    # from sklearn.model_selection import TimeSeriesSplit
    # splitter = TimeSeriesSplit(n_splits=5)
    # for tr, te in splitter.split(X_b, y_b): ...  (group by day, not by store)
print("scenario data ready - write your splitter below")

scenario data ready - write your splitter below


In [11]:
# Your code

**Justification.** For your scenario, what happens if the wrong splitter is used (plain KFold)? Quantify the expected effect on the reported scores (overly optimistic by how much, roughly)?


**Your answer:**

## Takeaways

- A complete ML workflow is a fixed sequence: problem -> data inspection -> baseline -> preprocessing (no leakage) -> model candidates -> valid CV -> metrics -> error analysis -> conclusions and limitations.
- Every step needs an explicit justification; the baseline and the CV scheme define the reference against which everything else is measured.
- Metrics must match the task and its costs: with imbalanced classes, accuracy is a poor summary and recall/F1/ROC-AUC carry the real information.
- Error analysis focuses on where the model is wrong or most uncertain; in high-performance settings this means borderline samples near the decision boundary.
- The 11-step mini-project checklist is the grading contract: missing steps (especially baseline, valid CV, error analysis, limitations) cost more than a slightly lower score.
- For grouped or time-structured data, the validation scheme is a methodological decision, not a default: GroupKFold for patients, TimeSeriesSplit for time, and the choice must be justified in the report.